# DLAI Model Merging - Example-level error analysis

This notebook stops hyperparameter search and asks a more concrete question: on which real texts does merging preserve, lose, or recover a specialist's competence? It compares each seed-42 specialist with Mean and frozen TIES (density 0.2) across all six task pairs.

The analysis is explanatory, not a new confirmatory performance test. Attach notebook 02's Output containing `pilot_specialists_seed42.zip`, select **GPU T4 x2**, enable Internet, and Run All.

In [ ]:
!nvidia-smi
!find /kaggle/input -maxdepth 3 -type f | head -50

## Install project and extract specialists

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path
REPO='https://github.com/LeuxLello/Dlai-model-merging.git'; BRANCH='codex/multiseed-results'
WORKDIR=Path('/kaggle/working/Dlai-model-merging')
if WORKDIR.exists(): shutil.rmtree(WORKDIR)
subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,REPO,str(WORKDIR)])
subprocess.check_call([sys.executable,'-m','pip','install','-q','-e',str(WORKDIR)])
sys.path.insert(0,str(WORKDIR/'src')); os.chdir(WORKDIR)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(); print('Commit:',COMMIT)
bundles=list(Path('/kaggle/input').rglob('pilot_specialists_seed42.zip'))
assert bundles, 'Attach notebook-02 Output containing pilot_specialists_seed42.zip.'
ROOT=Path('/kaggle/working/pilot_specialists_seed42')
if ROOT.exists(): shutil.rmtree(ROOT)
with zipfile.ZipFile(bundles[0]) as archive: archive.extractall(ROOT)
assert len(list(ROOT.rglob('encoder.pt')))==4

## Load real datasets, states, and evaluators

In [ ]:
import itertools, json, platform
import numpy as np, pandas as pd, torch
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification
from dlai_merge.data import get_task
from dlai_merge.evaluation import TaskEvaluator
from dlai_merge.merging import mean_merge, ties_merge
assert torch.cuda.is_available(), 'Enable GPU T4 x2.'
GPU=torch.cuda.get_device_name(0); assert torch.cuda.get_device_capability(0)[0]>=7
TASKS=['sst2','imdb','mrpc','rte']; PAIRS=list(itertools.combinations(TASKS,2))
BASE='prajjwal1/bert-mini'; SEED=42; MAX_EXAMPLES=500
def artifact(task,name):
    found=list(ROOT.rglob(f'{task}/seed-{SEED}/{name}')); assert len(found)==1; return found[0]
encoders={t:torch.load(artifact(t,'encoder.pt'),map_location='cpu',weights_only=True) for t in TASKS}
heads={t:torch.load(artifact(t,'head.pt'),map_location='cpu',weights_only=True) for t in TASKS}
base_model=AutoModelForSequenceClassification.from_pretrained(BASE,num_labels=2)
base_encoder={k:v.detach().cpu().clone() for k,v in base_model.base_model.state_dict().items()}
evaluators={t:TaskEvaluator(t,heads[t],max_eval_samples=MAX_EXAMPLES,seed=SEED,output_root='/kaggle/working/error-eval') for t in TASKS}
raw_eval={}; dataset_rows=[]
for task in TASKS:
    spec=get_task(task); raw=load_dataset(spec.dataset_name,spec.dataset_config)
    split=raw[spec.validation_split]
    if len(split)>MAX_EXAMPLES: split=split.shuffle(seed=SEED).select(range(MAX_EXAMPLES))
    raw_eval[task]=split
    labels=np.asarray(split['label']); counts={int(k):int(v) for k,v in zip(*np.unique(labels,return_counts=True))}
    dataset_rows.append({'task':task,'hf_dataset':spec.dataset_name,'config':spec.dataset_config or '',
      'evaluation_split':spec.validation_split,'available_split_rows':len(raw[spec.validation_split]),
      'analyzed_rows':len(split),'text_columns':'+'.join(spec.text_columns),'label_counts':json.dumps(counts)})
dataset_audit=pd.DataFrame(dataset_rows); display(dataset_audit)

## Prediction transitions
`preserved_correct` means both specialist and merge are correct; `merge_loss` means the specialist was correct and the merge became wrong; `merge_gain` is the reverse; `shared_failure` means both are wrong.

In [ ]:
def predict(task,state):
    evaluator=evaluators[task]; evaluator.model.base_model.load_state_dict(state,strict=True)
    output=evaluator.trainer.predict(evaluator.trainer.eval_dataset)
    logits=np.asarray(output.predictions); labels=np.asarray(output.label_ids)
    probs=torch.softmax(torch.tensor(logits),dim=1).numpy(); preds=probs.argmax(axis=1)
    return labels,preds,probs.max(axis=1)
def transition(specialist_ok,merged_ok):
    if specialist_ok and merged_ok: return 'preserved_correct'
    if specialist_ok and not merged_ok: return 'merge_loss'
    if not specialist_ok and merged_ok: return 'merge_gain'
    return 'shared_failure'
specialist_predictions={t:predict(t,encoders[t]) for t in TASKS}
rows=[]
for pair in PAIRS:
    states=[encoders[t] for t in pair]
    merged_states={'mean':mean_merge(base_encoder,states),'ties':ties_merge(base_encoder,states,density=0.2,scale=1.0)}
    for task in pair:
        labels,spec_pred,spec_conf=specialist_predictions[task]; raw=raw_eval[task]; spec=get_task(task)
        assert len(labels)==len(raw)
        for method,state in merged_states.items():
            labels2,merged_pred,merged_conf=predict(task,state); assert np.array_equal(labels,labels2)
            for i in range(len(labels)):
                item=raw[int(i)]; text_a=str(item[spec.text_columns[0]]); text_b=str(item[spec.text_columns[1]]) if len(spec.text_columns)>1 else ''
                specialist_ok=bool(spec_pred[i]==labels[i]); merged_ok=bool(merged_pred[i]==labels[i])
                rows.append({'pair':'+'.join(pair),'task':task,'method':method,'example_index':i,
                  'text_a':text_a,'text_b':text_b,'label':int(labels[i]),'specialist_prediction':int(spec_pred[i]),
                  'merged_prediction':int(merged_pred[i]),'specialist_confidence':float(spec_conf[i]),
                  'merged_confidence':float(merged_conf[i]),'transition':transition(specialist_ok,merged_ok)})
predictions=pd.DataFrame(rows); assert len(predictions)==2*3*sum(len(raw_eval[t]) for t in TASKS)
print('Example-method rows:',len(predictions))

## Aggregate patterns and readable examples

In [ ]:
transition_counts=(predictions.groupby(['pair','task','method','transition']).size().rename('count').reset_index())
totals=transition_counts.groupby(['pair','task','method'])['count'].transform('sum'); transition_counts['rate']=transition_counts['count']/totals
accuracy_summary=(predictions.assign(specialist_correct=lambda x:x.specialist_prediction==x.label,merged_correct=lambda x:x.merged_prediction==x.label)
 .groupby(['pair','task','method'],as_index=False).agg(n=('label','size'),specialist_accuracy=('specialist_correct','mean'),merged_accuracy=('merged_correct','mean')))
accuracy_summary['accuracy_delta']=accuracy_summary.merged_accuracy-accuracy_summary.specialist_accuracy
losses=(predictions[predictions.transition=='merge_loss'].sort_values(['pair','task','method','merged_confidence'],ascending=[True,True,True,False]).groupby(['pair','task','method']).head(5))
gains=(predictions[predictions.transition=='merge_gain'].sort_values(['pair','task','method','merged_confidence'],ascending=[True,True,True,False]).groupby(['pair','task','method']).head(5))
display(accuracy_summary.sort_values('accuracy_delta')); display(transition_counts)

## Figures and export

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
FIG=Path('/kaggle/working/error_analysis_figures'); FIG.mkdir(exist_ok=True)
fig,axes=plt.subplots(1,2,figsize=(14,5))
sns.barplot(data=accuracy_summary,x='pair',y='accuracy_delta',hue='method',ax=axes[0]); axes[0].axhline(0,color='black',lw=1); axes[0].tick_params(axis='x',rotation=25); axes[0].set_title('Accuracy change from specialist')
ties_trans=transition_counts[transition_counts.method=='ties']; sns.barplot(data=ties_trans,x='pair',y='rate',hue='transition',ax=axes[1]); axes[1].tick_params(axis='x',rotation=25); axes[1].set_title('TIES prediction transitions')
fig.tight_layout(); figure=FIG/'example_level_error_analysis.png'; fig.savefig(figure,dpi=180,bbox_inches='tight'); plt.show()
OUT=Path('/kaggle/working/example_level_error_analysis_results'); OUT.mkdir(exist_ok=True)
dataset_audit.to_csv(OUT/'dataset_audit.csv',index=False); predictions.to_csv(OUT/'all_predictions.csv',index=False)
accuracy_summary.to_csv(OUT/'accuracy_summary.csv',index=False); transition_counts.to_csv(OUT/'transition_counts.csv',index=False)
losses.to_csv(OUT/'representative_merge_losses.csv',index=False); gains.to_csv(OUT/'representative_merge_gains.csv',index=False)
metadata={'purpose':'seed-42 explanatory example-level error analysis','commit':COMMIT,'base_model':BASE,'tasks':TASKS,'pairs':['+'.join(p) for p in PAIRS],'seed':SEED,'max_examples_per_task':MAX_EXAMPLES,'methods':['mean','ties'],'ties_density':0.2,'gpu':GPU,'python':platform.python_version(),'torch':torch.__version__,'confirmatory_claim':False}
(OUT/'metadata.json').write_text(json.dumps(metadata,indent=2)); shutil.copy2(figure,OUT/figure.name)
archive=shutil.make_archive('/kaggle/working/example_level_error_analysis_results','zip',OUT); print(archive)